In [1]:
import os
import numpy as np
import pandas as pd

RAW_DIR = "../data/raw"
PROC_DIR = "../data/processed"

VALUE_PATH = os.path.join(PROC_DIR, "player_total_value_2025_26.csv")
SALARY_PATH = os.path.join(RAW_DIR, "salaries_2025_26.csv")

df = pd.read_csv(VALUE_PATH)
salaries = pd.read_csv(SALARY_PATH)

print("player value:", df.shape)
print("salaries:", salaries.shape)

salaries.head()

player value: (548, 33)
salaries: (471, 11)


,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player-additional
0,1,Stephen Curry,Golden State Warriors,$59606817,$62587158,NaN,NaN,NaN,NaN,$122193975,curryst01
1,2,Joel Embiid,Philadelphia 76ers,$55224526,$58100000,$62748000,$67396000,NaN,NaN,$176072526,embiijo01
2,3,Nikola Jokic,Denver Nuggets,$55224526,$59033114,$62841702,NaN,NaN,NaN,$114257640,jokicni01
3,4,Kevin Durant,Houston Rockets,$54708609,$43902439,$46097561,NaN,NaN,NaN,$98611048,duranke01
4,5,Jayson Tatum,Boston Celtics,$54126450,$58456566,$62786682,$67116798,$71446914,NaN,$242486496,tatumja01


In [2]:
salaries = salaries.rename(columns={
    "Player": "player_name",
    "Tm": "team",
    "2025-26": "salary"
})

# strip whitespace
for col in ["player_name", "team"]:
    salaries[col] = salaries[col].astype(str).str.strip()

# convert "$12,345,678" -> 12345678
salaries["salary"] = (
    salaries["salary"]
    .astype(str)
    .replace(r"[\$,]", "", regex=True)
)

salaries["salary"] = pd.to_numeric(salaries["salary"], errors="coerce").fillna(0)

salaries[["player_name","team","salary"]].head()

,player_name,team,salary
0,Stephen Curry,Golden State Warriors,59606817.0
1,Joel Embiid,Philadelphia 76ers,55224526.0
2,Nikola Jokic,Denver Nuggets,55224526.0
3,Kevin Durant,Houston Rockets,54708609.0
4,Jayson Tatum,Boston Celtics,54126450.0


In [3]:
# strip here too
for col in ["player_name", "team"]:
    df[col] = df[col].astype(str).str.strip()

df = df.merge(
    salaries[["player_name","team","salary"]],
    on=["player_name","team"],
    how="left"
)

# if salary missing, set to 0
df["salary"] = df["salary"].fillna(0)

print("missing salary rows:", (df["salary"] == 0).sum())
df[["player_name","team","salary","total_value"]].head()

missing salary rows: 98


,player_name,team,salary,total_value
0,Nikola Jokic,Denver Nuggets,55224526.0,10749.530071
1,Kawhi Leonard,Los Angeles Clippers,50000000.0,9905.979792
2,Shai Gilgeous-Alexander,Oklahoma City Thunder,38333050.0,13121.279938
3,Giannis Antetokounmpo,Milwaukee Bucks,54126450.0,6999.240217
4,Donovan Mitchell,Cleveland Cavaliers,46394100.0,12002.800155


In [4]:
df["value_per_million"] = np.where(
    df["salary"] > 0,
    df["total_value"] / (df["salary"] / 1_000_000),
    np.nan
)

# Filter to stable rotation minutes for rankings
df_filtered = df[df["minutes"] >= 500].copy()

top15 = df_filtered.dropna(subset=["value_per_million"]).sort_values("value_per_million", ascending=False).head(15)
bottom15 = df_filtered.dropna(subset=["value_per_million"]).sort_values("value_per_million", ascending=True).head(15)

top15[["player_name","team","salary","minutes","value_per_million"]], bottom15[["player_name","team","salary","minutes","value_per_million"]]

(               player_name                    team     salary  minutes  \
 54        Collin Gillespie            Phoenix Suns  2296274.0   1413.0   
 120         Toumani Camara  Portland Trail Blazers  2221677.0   1721.0   
 73           Neemias Queta          Boston Celtics  2349578.0   1220.0   
 77   Sandro Mamukelashvili         Toronto Raptors  2461463.0   1135.0   
 103         Moussa Diabate       Charlotte Hornets  2270735.0   1177.0   
 95       Julian Champagnie       San Antonio Spurs  3000000.0   1427.0   
 123           Oso Ighodaro            Phoenix Suns  1955377.0   1020.0   
 104            Josh Okogie         Houston Rockets  2296274.0    979.0   
 93            Ryan Rollins         Milwaukee Bucks  4000000.0   1566.0   
 183             A.J. Green         Milwaukee Bucks  2301587.0   1343.0   
 94          Mouhamed Gueye           Atlanta Hawks  2221677.0    824.0   
 105           Jordan Walsh          Boston Celtics  2221677.0    844.0   
 114     Brandin Podziems

In [5]:
OUT_PATH = os.path.join(PROC_DIR, "player_cost_efficiency_2025_26.csv")
df_filtered.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Saved: ../data/processed/player_cost_efficiency_2025_26.csv


In [6]:
print("Players with minutes >= 500:", df_filtered.shape[0])

Players with minutes >= 500: 320
